In [67]:
"""
Contextual Compression Retriever RAG Pipeline -- Production-Grade
=================================================================
Architecture   : FIVE configurations covering every compression strategy in
                 LangChain's ContextualCompressionRetriever ecosystem:
                 (1) Standard RAG Baseline           -- no compression, raw retrieval
                 (2) LLMChainExtractor               -- LLM extracts relevant spans
                 (3) LLMChainFilter + LLMListwiseRerank -- drop irrelevant docs entirely
                 (4) EmbeddingsFilter                -- embedding-similarity filter (fast)
                 (5) DocumentCompressorPipeline      -- 4-stage orchestrated pipeline [BEST]
                     [TextSplitter -> EmbeddingsRedundantFilter ->
                      EmbeddingsFilter -> LLMChainExtractor]

Vector Store   : FAISS (IndexFlatIP, BGE-large-en-v1.5, 1024-dim, retrieve_k=10)
BM25           : BM25Plus (rank-bm25) -- Hybrid base retriever for Configs 2-5
Embeddings     : BAAI/bge-large-en-v1.5  (1024-dim, query instruction prefix)
LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+  (ContextualCompressionRetriever,
                                  DocumentCompressorPipeline, EnsembleRetriever)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
The Context Overflow Problem in Standard RAG
=============================================================================
Every LLM has a finite context window. Even models with large windows (128K+)
suffer from the "lost-in-the-middle" degradation: when relevant information is
buried in a long context surrounded by irrelevant text, the LLM's attention
mechanism assigns it lower weight than information at the start or end of the
context. Performance is systematically worse when relevant information appears
in the middle of the context.

Standard RAG exacerbates this in two interconnected ways:

    1. CHUNK NOISE: A retrieved chunk is typically 300-600 tokens of text. The
       relevant answer to the query may span only 1-2 sentences (30-80 tokens)
       within the chunk. The remaining 85-90% of the chunk is context that was
       necessary for the original passage but is irrelevant to the specific query.
       When 10 chunks are retrieved, only ~10-20% of the total token budget
       actually contains query-relevant content. The LLM reads 5-10x more tokens
       than necessary to answer the question.

    2. INTER-DOCUMENT REDUNDANCY: Vector similarity search retrieves the top-K
       most semantically similar chunks. In a corpus of research papers, these
       top-K chunks frequently cover the SAME section of the paper from adjacent
       (overlapping) chunk windows. Multiple chunks about the same table or
       paragraph are near-duplicates that consume context budget without adding
       information. This is especially severe with chunking strategies that use
       high overlap ratios (>30%).

Context overflow is one of the biggest challenges in any RAG-based application.
The challenge with retrieval is that usually you don't know the specific document
relevant to the query at the time of indexing, making it hard to set the right
chunk size. Contextual compression solves this by dynamically trimming each chunk
to only its query-relevant content AFTER retrieval and BEFORE feeding to the LLM.

=============================================================================
What Contextual Compression Is
=============================================================================
The ContextualCompressionRetriever wraps any base retriever with a compressor.
The flow is:

    query -> base_retriever.invoke(query) -> [D_1, D_2, ..., D_k] (raw chunks)
          -> compressor.compress_documents([D_1..D_k], query)
          -> [C_1, C_2, ..., C_m]  (compressed/filtered documents, m <= k)
          -> LLM reads [C_1..C_m] instead of [D_1..D_k]

The compressor has two modes:
    EXTRACT MODE: Rewrite each document to keep only relevant content.
                  Each output document is SHORTER than the input.
                  Output set size == Input set size (no documents dropped,
                  just compressed).
    FILTER MODE:  Drop entire documents that are not relevant.
                  Output set size <= Input set size (documents dropped).
                  Content of retained documents is UNCHANGED.

These modes can be chained in a DocumentCompressorPipeline:
    Extract then Filter -> compress content, then drop low-relevance remainder.
    Filter then Extract -> remove irrelevant docs first, compress survivors.
    Split then Filter  -> fine-grain chunks, filter sub-chunks by relevance.

=============================================================================
The Five Compressor Architectures
=============================================================================

(1) BASELINE: No compression. Raw hybrid retrieval. Establishes context
              length and answer quality floor.

(2) LLMChainExtractor: Calls the LLM for EACH document independently.
    Prompt: "Given the question, extract verbatim relevant statements from
    this document. If none are relevant, return NO_OUTPUT."
    Result: Each chunk is reduced to only its query-relevant sentences.
    Typical compression ratio: 2x-5x token reduction per document.
    Cost: K LLM calls (one per retrieved doc). Expensive but high quality.

(3) LLMChainFilter + LLMListwiseRerank:
    LLMChainFilter: Simpler -- calls LLM per doc to decide yes/no (keep/drop).
    Does NOT modify document content. Binary relevance classification.
    LLMListwiseRerank: Zero-shot listwise reranking -- LLM sees ALL docs at once
    and ranks them, returning only the top_n. Requires with_structured_output.
    Combined: Filter drops irrelevant docs, then Listwise reranks survivors.

(4) EmbeddingsFilter: No LLM calls. Embeds each document and the query.
    Computes cosine similarity. Drops documents below similarity_threshold.
    Fastest compression option. No per-doc LLM cost. Zero extraction quality.
    Typical threshold: 0.76 (stricter) to 0.60 (more permissive).
    Best suited for: high-volume, latency-sensitive production systems.

(5) DocumentCompressorPipeline [BEST]: Four-stage pipeline:
    Stage 1 -- RecursiveCharacterTextSplitter (chunk_size=200):
        Re-splits each retrieved chunk into fine-grained sub-chunks.
        Enables sentence-level relevance filtering in stages 2-4.
        Without this, EmbeddingsFilter operates on full 450-char chunks
        where irrelevant sentences dilute the embedding similarity score.
    Stage 2 -- EmbeddingsRedundantFilter (similarity_threshold=0.95):
        Removes near-duplicate sub-chunks across ALL retrieved documents.
        Two sub-chunks with embedding cosine > 0.95 = one is dropped.
        Eliminates the inter-document redundancy problem from chunking overlap.
    Stage 3 -- EmbeddingsFilter (similarity_threshold=0.72):
        Drops sub-chunks with cosine similarity to query below 0.72.
        Fast (no LLM), eliminates the majority of off-topic sub-chunks.
        Typical reduction: 60-80% of sub-chunks dropped here.
    Stage 4 -- LLMChainExtractor:
        Applied to the small set of survivors from Stage 3.
        LLM cost is now manageable because Stage 3 already dropped most content.
        Extracts ONLY the verbatim relevant statements from each surviving sub-chunk.
    Net result: 10x-20x context reduction. Only query-relevant sentences reach the LLM.
=====================================================================
"""

'\nContextual Compression Retriever RAG Pipeline -- Production-Grade\n=================================================================\nArchitecture   : FIVE configurations covering every compression strategy in\n                 LangChain\'s ContextualCompressionRetriever ecosystem:\n                 (1) Standard RAG Baseline           -- no compression, raw retrieval\n                 (2) LLMChainExtractor               -- LLM extracts relevant spans\n                 (3) LLMChainFilter + LLMListwiseRerank -- drop irrelevant docs entirely\n                 (4) EmbeddingsFilter                -- embedding-similarity filter (fast)\n                 (5) DocumentCompressorPipeline      -- 4-stage orchestrated pipeline [BEST]\n                     [TextSplitter -> EmbeddingsRedundantFilter ->\n                      EmbeddingsFilter -> LLMChainExtractor]\n\nVector Store   : FAISS (IndexFlatIP, BGE-large-en-v1.5, 1024-dim, retrieve_k=10)\nBM25           : BM25Plus (rank-bm25) -- Hybrid bas

In [68]:
import os 
import re 
import sys 
import time 
import pickle 
import logging 
import urllib.request 
import dataclasses
from dataclasses import dataclass, field 
from pathlib import Path 
from typing import List, Dict, Any, Optional, Tuple, Callable

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS 
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import EnsembleRetriever , ContextualCompressionRetriever
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter, LLMChainExtractor, LLMChainFilter, LLMListwiseRerank, DocumentCompressorPipeline
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser

In [69]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("contextual_compression_rag")


In [70]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class CCConfig:
    """
    Centralized configuration for the Contextual Compression Retriever RAG pipeline.

    BASE_RETRIEVE_K (10) -- Why retrieve 10 before compressing to 4:
        The compression pipeline acts as a precision filter. If we only retrieve
        K=4 initially, the compression step may reduce us to 1-2 documents
        (insufficient context for the LLM). By retrieving K=10 and letting the
        compressor trim to the best 4-6, we:
            - Maximize recall at the retrieval stage (higher initial K).
            - Maximize precision at the generation stage (compression removes noise).
        This is the "over-retrieve then compress" pattern, which is the production
        standard for contextual compression pipelines.

    EMBEDDINGS_FILTER_THRESHOLD (0.72) -- Calibration:
        EmbeddingsFilter uses cosine similarity of the sub-chunk embedding
        vs the query embedding. The production-validated threshold range is 0.60-0.80:
            < 0.60: Too permissive. Nearly all sub-chunks pass. No effective filtering.
            0.60-0.72: Moderate filtering. Good for broad queries.
            0.72-0.80: Strict filtering. Best for narrow technical queries.
            > 0.80: Over-filtering. Risk of dropping relevant sub-chunks.
        0.72 is the production default for technical domain corpora.
        CRITICAL: This threshold is embedding-model-dependent. A threshold calibrated
        for OpenAI ada-002 CANNOT be transferred to BGE-large without recalibration.
        BGE-large normalized embeddings have a different cosine distribution than ada-002.

    REDUNDANCY_THRESHOLD (0.95):
        EmbeddingsRedundantFilter drops a sub-chunk if its cosine similarity
        to any already-seen sub-chunk exceeds 0.95. This is effectively
        "deduplication at 95% similarity". Two sub-chunks at 0.95 cosine similarity
        are near-identical in content -- keeping both provides no additional context
        to the LLM while consuming valuable context budget.
        Setting this too low (e.g., 0.80) would incorrectly drop sub-chunks that
        discuss related but distinct aspects of the same technical concept.

    PIPELINE_SPLIT_CHUNK_SIZE (200):
        The DocumentCompressorPipeline's first stage re-splits retrieved chunks into
        200-character sub-chunks before filtering. This is smaller than the
        index-time chunk size (450 chars) for a critical reason:
            - Index chunk size = 450 chars for RECALL: large chunks contain enough
              context to be found by vector search.
            - Pipeline sub-chunk size = 200 chars for PRECISION: fine-grained sub-chunks
              isolate individual sentences/facts for relevance filtering.
        Without this re-splitting, EmbeddingsFilter operates on 450-char chunks
        that mix relevant and irrelevant sentences, causing false negatives (relevant
        chunks dropped because irrelevant sentences dilute the embedding similarity).

    LLM_COMPRESS_TEMPERATURE (0.0):
        LLM-based compressors (LLMChainExtractor, LLMChainFilter) must use
        temperature=0.0. Compression is a PRECISION task: the output must be a
        faithful extraction of verbatim content from the input document.
        Temperature > 0 risks generating paraphrased or hallucinated "extractions"
        that are not present in the source document.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",     "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers",  "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",   "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- FAISS -----------------------------------------------------------
    FAISS_PERSIST_DIR: str = "./faiss_cc_index"

    # --- BM25 ------------------------------------------------------------
    BM25_PERSIST_DIR: str  = "./bm25_cc_index"
    BM25_PARAMS:      Dict = {"k1": 1.5, "b": 0.75, "delta": 0.5}

    # --- BGE Embeddings --------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Retrieval K Parameters ------------------------------------------
    BASE_RETRIEVE_K:  int = 10   # chunks retrieved by base retriever before compression
    BM25_K:           int = 10   # BM25 candidates for hybrid base retriever
    DENSE_K:          int = 10   # dense FAISS candidates for hybrid base retriever
    FINAL_K:          int = 4    # target documents after compression

    # --- Hybrid RRF Weights -----------------------------------------------
    HYBRID_WEIGHTS: List[float] = [0.4, 0.6]  # [bm25, dense]

    # --- Compression Parameters ------------------------------------------
    EMBEDDINGS_FILTER_THRESHOLD: float = 0.72  # EmbeddingsFilter cosine threshold
    REDUNDANCY_THRESHOLD:        float = 0.95  # EmbeddingsRedundantFilter threshold
    PIPELINE_SPLIT_CHUNK_SIZE:   int   = 200   # sub-chunk size inside the pipeline
    LISTWISE_TOP_N:              int   = 4     # top_n for LLMListwiseRerank
    LLM_COMPRESS_TEMPERATURE:   float = 0.0   # temperature for compression LLM calls
    LLM_ANSWER_TEMPERATURE:     float = 0.0   # temperature for answer LLM calls

    # --- Text Splitting (index-time) -------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Azure OpenAI LLM ------------------------------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int            = 1024

    # --- RAG Answer Prompt -----------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Context (contextually compressed):
{context}

Question: {question}

Precise technical answer:"""



In [71]:

# ===========================================================================
# SECTION 2 -- COMPRESSION TRACE DATA CLASS
# ===========================================================================

@dataclass
class CompressionTrace:
    """
    Records the complete execution trace of a contextual compression retrieval run.

    The compression_ratio is the primary observability metric for this pipeline:
        compression_ratio = total_input_chars / total_output_chars
        A ratio of 5.0 means the LLM reads 5x fewer tokens than standard RAG.
        Production target: compression_ratio >= 3.0 for meaningful benefit.
        If compression_ratio < 1.5: compressor is not triggering effectively.
            Cause: threshold too low (EmbeddingsFilter), or all docs already relevant.
        If compression_ratio > 20x: over-compression risk.
            Cause: threshold too high, dropping relevant content.

    per_doc_compression: per-document character count before and after compression.
        Input:  List[int] = character count of each raw retrieved chunk.
        Output: List[int] = character count of each compressed chunk (0 if dropped).
        Documents dropped by filtering have 0 in the output list.
    """
    question:             str
    strategy:             str
    raw_docs:             List[Document] = field(default_factory=list)
    compressed_docs:      List[Document] = field(default_factory=list)
    raw_char_counts:      List[int]      = field(default_factory=list)
    compressed_char_counts: List[int]    = field(default_factory=list)
    total_raw_chars:      int            = 0
    total_compressed_chars: int          = 0
    compression_ratio:    float          = 0.0
    docs_dropped:         int            = 0
    final_answer:         str            = ""
    retrieval_ms:         float          = 0.0
    compression_ms:       float          = 0.0
    generation_ms:        float          = 0.0
    total_ms:             float          = 0.0

    def compute_compression_metrics(self) -> None:
        """Compute compression ratio and per-document metrics from raw/compressed docs."""
        self.raw_char_counts = [len(d.page_content) for d in self.raw_docs]
        self.compressed_char_counts = [len(d.page_content) for d in self.compressed_docs]
        self.total_raw_chars = sum(self.raw_char_counts)
        self.total_compressed_chars = sum(self.compressed_char_counts)
        self.compression_ratio = (
            self.total_raw_chars / max(self.total_compressed_chars, 1)
        )
        # Docs dropped = raw docs that are not present in compressed docs
        self.docs_dropped = max(0, len(self.raw_docs) - len(self.compressed_docs))

    def print_trace(self) -> None:
        print(f"\n{'='*72}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:80]}")
        print(f"{'='*72}")

        print(f"\n  Raw docs: {len(self.raw_docs)}  ({self.total_raw_chars:,} chars)")
        print(f"  Compressed docs: {len(self.compressed_docs)} "
              f"({self.total_compressed_chars:,} chars)")
        print(f"  Docs dropped: {self.docs_dropped}")
        print(f"  Compression ratio: {self.compression_ratio:.2f}x")

        print(f"\n  Per-document compression:")
        for i, doc in enumerate(self.compressed_docs[:6]):
            paper = doc.metadata.get("paper_name", "?")[:28]
            page  = doc.metadata.get("page", "?")
            raw_c = self.raw_char_counts[i] if i < len(self.raw_char_counts) else "?"
            cmp_c = len(doc.page_content)
            ratio = raw_c / max(cmp_c, 1) if isinstance(raw_c, int) else 0.0
            print(
                f"    [{i+1}] {paper} p{page}: "
                f"{raw_c} -> {cmp_c} chars  ({ratio:.1f}x)"
            )

        print(f"\n  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"compress={self.compression_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:500]}")
        print("=" * 72 + "\n")


In [72]:

# ===========================================================================
# SECTION 3 -- PDF DOWNLOADER AND CHUNKER
# ===========================================================================

def download_pdfs(config: CCConfig) -> List[Path]:
    """Download research PDFs with file-existence caching."""
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-CC-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. Place manually at '{dest}'."
            ) from exc

    return paths


def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: CCConfig,
) -> List[Document]:
    """Load PDFs and split into chunks with paper_name and page metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []

    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})

        chunks = splitter.split_documents(pages)
        logger.info(
            "  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks)
        )
        all_chunks.extend(chunks)

    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks



In [73]:

# ===========================================================================
# SECTION 4 -- INDEX AND MODEL BUILDERS
# ===========================================================================

def _bm25_preprocess(text: str) -> List[str]:
    """Tokenization helper kept at module scope so BM25 cache can be pickled."""
    return text.lower().split()


def build_bge_embeddings(config: CCConfig) -> HuggingFaceEmbeddings:
    """Initialize BGE-large-en-v1.5 bi-encoder."""
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    # query_instruction is unsupported in this langchain_community version.
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss_index(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: CCConfig,
) -> FAISS:
    """Build or load FAISS index with persistence."""
    faiss_file = Path(config.FAISS_PERSIST_DIR) / "index.faiss"

    if faiss_file.exists():
        logger.info("Loading FAISS from '%s' ...", config.FAISS_PERSIST_DIR)
        try:
            vs = FAISS.load_local(
                config.FAISS_PERSIST_DIR, embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info("FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)

    logger.info("Building FAISS from %d chunks ...", len(chunks))
    t0 = time.perf_counter()
    vs = FAISS.from_documents(chunks, embeddings)
    logger.info(
        "FAISS built. Vectors: %d  (%.2fs)", vs.index.ntotal, time.perf_counter() - t0
    )
    Path(config.FAISS_PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_PERSIST_DIR)
    return vs


def build_bm25_index(
    chunks: List[Document],
    config: CCConfig,
) -> BM25Retriever:
    """Build or load BM25 retriever (compatible with BM25Okapi-only installs)."""
    cache = Path(config.BM25_PERSIST_DIR) / "bm25plus.pkl"

    if cache.exists():
        logger.info("Loading BM25 from '%s' ...", cache)
        try:
            with open(cache, "rb") as f:
                state = pickle.load(f)
            ret = BM25Retriever(
                vectorizer=state["vectorizer"], docs=state["docs"],
                k=state["k"], preprocess_func=state["preprocess_func"],
            )
            return ret
        except Exception as exc:
            logger.warning("BM25 cache load failed (%s). Rebuilding index ...", exc)
            try:
                cache.unlink(missing_ok=True)
            except Exception:
                pass

    logger.info("Building BM25 from %d chunks ...", len(chunks))
    bm25_params = dict(config.BM25_PARAMS)
    # Older rank_bm25/langchain paths use BM25Okapi and reject BM25Plus-only params.
    if "delta" in bm25_params:
        logger.warning(
            "BM25 parameter 'delta' is unsupported by BM25Okapi in this environment; dropping it."
        )
        bm25_params.pop("delta")

    ret = BM25Retriever.from_documents(
        chunks,
        bm25_params=bm25_params,
        preprocess_func=_bm25_preprocess,
    )
    ret.k = config.BM25_K
    cache.parent.mkdir(parents=True, exist_ok=True)
    with open(cache, "wb") as f:
        pickle.dump(
            {"vectorizer": ret.vectorizer, "docs": ret.docs,
             "k": ret.k, "preprocess_func": ret.preprocess_func},
            f, protocol=pickle.HIGHEST_PROTOCOL,
        )
    logger.info("BM25 persisted to '%s'", cache)
    return ret


def build_ollama_llm(
    config: CCConfig,
    temperature: float = None,
) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )
    return ChatOllama(
        base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
        model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
        temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )


def build_hybrid_base_retriever(
    vectorstore: FAISS,
    bm25: BM25Retriever,
    config: CCConfig,
) -> EnsembleRetriever:
    """
    Build BM25+Dense RRF hybrid base retriever for contextual compression.

    Why hybrid as the BASE retriever for compression:
        Contextual compression is a post-retrieval operation. The quality of
        compression output is bounded by the quality of what the base retriever
        fetches. A hybrid base retriever has higher recall than either BM25 or
        dense retrieval alone, ensuring the compressor has the best possible
        candidate set to work with.

        Using only dense retrieval as the base is the most common mistake in
        compression pipeline setups. For technical queries with precise terminology
        ("BLEU score WMT 2014"), BM25 has significantly better recall than dense
        retrieval. Supplying these BM25 results to the compressor gives the
        extractor the exact sentences it needs to extract.
    """
    bm25_r = BM25Retriever(
        vectorizer=bm25.vectorizer, docs=bm25.docs,
        k=config.BM25_K, preprocess_func=bm25.preprocess_func,
    )
    dense_r = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.DENSE_K},
    )
    return EnsembleRetriever(
        retrievers=[bm25_r, dense_r],
        weights=config.HYBRID_WEIGHTS,
    )


In [74]:


# ===========================================================================
# SECTION 5 -- SHARED UTILITIES
# ===========================================================================

def format_docs(docs: List[Document]) -> str:
    """Format documents into annotated context block for LLM answer generation."""
    parts = []
    for i, doc in enumerate(docs, 1):
        paper = doc.metadata.get("paper_name", "Unknown")
        page  = doc.metadata.get("page", "?")
        chars = len(doc.page_content)
        parts.append(
            f"[{i} | {paper} | p{page} | {chars} chars]\n"
            f"{doc.page_content.strip()}"
        )
    return ("\n\n" + "-" * 60 + "\n\n").join(parts)


def generate_answer(
    question: str,
    docs:     List[Document],
    llm:      ChatOllama,
    config:   CCConfig,
) -> Tuple[str, float]:
    """Generate answer from compressed documents. Returns (answer, latency_ms)."""
    context = format_docs(docs)
    prompt  = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    t0      = time.perf_counter()
    answer  = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return answer, (time.perf_counter() - t0) * 1000


def retrieve_raw_docs(
    question:  str,
    retriever: EnsembleRetriever,
    config:    CCConfig,
) -> Tuple[List[Document], float]:
    """Retrieve raw docs from base hybrid retriever. Returns (docs, latency_ms)."""
    t0   = time.perf_counter()
    docs = retriever.invoke(question)
    return docs[: config.BASE_RETRIEVE_K], (time.perf_counter() - t0) * 1000


In [75]:
from langchain_classic.retrievers.document_compressors import LLMListwiseRerank


In [76]:

# ===========================================================================
# SECTION 6 -- FIVE COMPRESSION CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Standard RAG Baseline
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question:    str,
    vectorstore: FAISS,
    bm25:        BM25Retriever,
    llm_answer:  ChatOllama,
    config:      CCConfig,
) -> CompressionTrace:
    """
    Configuration 1: Standard RAG Baseline -- no compression.

    Raw hybrid retrieval passes directly to LLM. Zero compression.
    Establishes raw context length and answer quality floor.
    The compression_ratio will be 1.0 by definition.
    """
    trace = CompressionTrace(question=question, strategy="Config1_Baseline_NoCompression")
    t0    = time.perf_counter()

    retriever = build_hybrid_base_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw_docs(question, retriever, config)

    trace.raw_docs        = raw_docs
    trace.compressed_docs = raw_docs[: config.FINAL_K]
    trace.compute_compression_metrics()

    answer, trace.generation_ms = generate_answer(question, trace.compressed_docs, llm_answer, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


# ---------------------------------------------------------------------------
# CONFIG 2: LLMChainExtractor
# ---------------------------------------------------------------------------

def run_config2_llm_chain_extractor(
    question:    str,
    vectorstore: FAISS,
    bm25:        BM25Retriever,
    llm_compress: ChatOllama,
    llm_answer:  ChatOllama,
    config:      CCConfig,
) -> CompressionTrace:
    """
    Configuration 2: LLMChainExtractor.

    LLMChainExtractor iterates over the initially returned documents and
    extracts from each only the content that is relevant to the query.
    The LLMChainExtractor uses an LLMChain to extract from each document
    only the statements that are relevant to the query.

    Internal LLMChainExtractor prompt (paraphrased from LangChain source):
        "Given the following question and context, extract any part of the
        context *AS IS* that is relevant to answer the question. If none of
        the context is relevant, return NO_OUTPUT."

    Design decisions:
        - llm_compress uses temperature=0.0 for deterministic extraction.
        - The extractor does NOT paraphrase or summarize. It extracts VERBATIM
          spans from the source document. This is critical for faithfulness:
          the LLM answer generation can later cite the exact source text.
        - Documents where no relevant content exists return an empty string
          or "NO_OUTPUT", which the ContextualCompressionRetriever drops.
        - LLM call count: BASE_RETRIEVE_K calls (one per raw document).
          At BASE_RETRIEVE_K=10, this is 10 LLM calls before the answer call.
          For production with high QPS, consider EmbeddingsFilter (Config 4)
          or the DocumentCompressorPipeline (Config 5) which pre-filters docs
          before LLM extraction.

    Typical observed behavior:
        - A 450-char chunk about multi-head attention architecture is compressed
          to the 80-120 chars that directly describe the computation mechanics
          relevant to the query "how does multi-head attention compute outputs."
        - Irrelevant passages (e.g., training hyperparameters retrieved from
          a general "transformer" query) are returned as empty and dropped.
        - Typical compression ratio: 3x-5x for well-typed queries.

    Args:
        question      : User query.
        vectorstore   : FAISS index.
        bm25          : BM25Plus retriever.
        llm_compress  : LLM for extraction (temperature=0.0).
        llm_answer    : LLM for answer generation (temperature=0.0).
        config        : CCConfig.

    Returns:
        CompressionTrace with per-doc extraction ratios and final answer.
    """
    trace = CompressionTrace(question=question, strategy="Config2_LLMChainExtractor")
    t0    = time.perf_counter()

    base_retriever = build_hybrid_base_retriever(vectorstore, bm25, config)

    # LLMChainExtractor: extracts relevant spans from each document
    extractor   = LLMChainExtractor.from_llm(llm_compress)
    cc_retriever = ContextualCompressionRetriever(
        base_compressor=extractor,
        base_retriever=base_retriever,
    )

    # Capture raw docs for compression ratio measurement
    raw_docs, trace.retrieval_ms = retrieve_raw_docs(question, base_retriever, config)
    trace.raw_docs = raw_docs

    # Run compression
    t_compress             = time.perf_counter()
    compressed_docs        = cc_retriever.invoke(question)
    trace.compression_ms   = (time.perf_counter() - t_compress) * 1000
    trace.compressed_docs  = compressed_docs[: config.FINAL_K]
    trace.compute_compression_metrics()

    logger.info(
        "Config2 LLMChainExtractor: %d raw -> %d compressed  (%.2fx ratio)",
        len(raw_docs), len(trace.compressed_docs), trace.compression_ratio,
    )

    answer, trace.generation_ms = generate_answer(question, trace.compressed_docs, llm_answer, config)
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


# ---------------------------------------------------------------------------
# CONFIG 3: LLMChainFilter + LLMListwiseRerank
# ---------------------------------------------------------------------------

def run_config3_llm_filter_listwise(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       CCConfig,
) -> CompressionTrace:
    """
    Configuration 3: LLMChainFilter followed by LLMListwiseRerank.

    TWO-STAGE FILTER ARCHITECTURE:

    Stage 1 -- LLMChainFilter:
        The LLMChainFilter is a slightly simpler but more robust compressor
        that uses an LLM chain to decide which of the initially retrieved
        documents to filter out and which ones to return, WITHOUT manipulating
        the document contents.
        Key difference from LLMChainExtractor: the document content is PRESERVED
        exactly as retrieved. The LLM only makes a binary keep/drop decision,
        not an extraction decision. This means:
            - No risk of LLM hallucinating or modifying source content.
            - Retained documents still contain noise (non-relevant sentences).
            - Better faithfulness (source-exact) but lower compression ratio.

    Stage 2 -- LLMListwiseRerank:
        LLMListwiseRerank uses zero-shot listwise document reranking and
        functions similarly to LLMChainFilter as a robust but more expensive
        option. It is recommended to use a more powerful LLM.
        Unlike LLMChainFilter (per-doc binary decision), LLMListwiseRerank
        sees ALL surviving documents simultaneously and ranks them holistically.
        Returns only the top_n=LISTWISE_TOP_N documents.

        Listwise vs. Pointwise reranking:
            POINTWISE (LLMChainFilter): Each document graded independently.
              Advantage: parallelizable, each doc graded in isolation.
              Disadvantage: cannot compare relevance BETWEEN documents.
              A document that is "somewhat relevant" gets the same binary
              treatment regardless of whether there are 5 more-relevant docs.
            LISTWISE (LLMListwiseRerank): All documents graded together.
              Advantage: comparative assessment. The LLM can say "Doc 3 is
              more relevant than Doc 1 because it specifically mentions X."
              Disadvantage: all docs must fit in one LLM context window.
              Best for: final precision selection among a filtered candidate set.

        Combined pipeline rationale:
            LLMChainFilter reduces 10 raw docs to ~4-6 relevant docs.
            LLMListwiseRerank then selects the best FINAL_K=4 from those ~6.
            This two-stage approach is more cost-efficient than running
            LLMListwiseRerank directly on all 10 docs.

    Implementation note on LLMListwiseRerank:
        LLMListwiseRerank requires a model with the with_structured_output
        method implemented. ChatOllama supports this.
        It is used as a standalone reranker here (not via ContextualCompressionRetriever)
        because the pipeline architecture is filter-then-rerank rather than
        a single ContextualCompressionRetriever call.

    Args:
        question      : User query.
        vectorstore   : FAISS index.
        bm25          : BM25Plus retriever.
        llm_compress  : LLM for both filter and listwise rerank (temperature=0.0).
        llm_answer    : LLM for answer generation (temperature=0.0).
        config        : CCConfig.

    Returns:
        CompressionTrace with per-doc filter and rerank metrics.
    """
    trace = CompressionTrace(
        question=question, strategy="Config3_LLMChainFilter+ListwiseRerank"
    )
    t0 = time.perf_counter()

    base_retriever = build_hybrid_base_retriever(vectorstore, bm25, config)

    raw_docs, trace.retrieval_ms = retrieve_raw_docs(question, base_retriever, config)
    trace.raw_docs = raw_docs
    logger.info("Config3: %d raw docs retrieved.", len(raw_docs))

    # Stage 1: LLMChainFilter -- binary keep/drop per document
    t_compress  = time.perf_counter()
    chain_filter = LLMChainFilter.from_llm(llm_compress)
    filter_retriever = ContextualCompressionRetriever(
        base_compressor=chain_filter,
        base_retriever=base_retriever,
    )
    filtered_docs = filter_retriever.invoke(question)
    logger.info(
        "Config3: LLMChainFilter kept %d/%d docs.",
        len(filtered_docs), len(raw_docs),
    )

    # Stage 2: LLMListwiseRerank -- listwise ranking of filtered survivors
    try:
        from langchain_classic.retrievers.document_compressors import LLMListwiseRerank

        if filtered_docs:
            listwise_reranker = LLMListwiseRerank.from_llm(
                llm_compress,
                top_n=config.LISTWISE_TOP_N,
            )
            reranked_docs = listwise_reranker.compress_documents(
                documents=filtered_docs,
                query=question,
            )
            logger.info(
                "Config3: LLMListwiseRerank selected top %d from %d filtered.",
                len(reranked_docs), len(filtered_docs),
            )
            final_docs = reranked_docs
        else:
            logger.warning(
                "Config3: LLMChainFilter dropped all docs. "
                "Falling back to raw top-%d.", config.FINAL_K
            )
            final_docs = raw_docs[: config.FINAL_K]

    except ImportError:
        logger.warning(
            "LLMListwiseRerank not available in this LangChain version. "
            "Using LLMChainFilter output directly."
        )
        final_docs = filtered_docs[: config.FINAL_K]

    trace.compression_ms  = (time.perf_counter() - t_compress) * 1000
    trace.compressed_docs = final_docs[: config.FINAL_K]
    trace.compute_compression_metrics()

    answer, trace.generation_ms = generate_answer(
        question, trace.compressed_docs, llm_answer, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace

In [77]:

# ---------------------------------------------------------------------------
# CONFIG 4: EmbeddingsFilter (Fast, LLM-Free)
# ---------------------------------------------------------------------------

def run_config4_embeddings_filter(
    question:    str,
    vectorstore: FAISS,
    bm25:        BM25Retriever,
    embeddings:  HuggingFaceEmbeddings,
    llm_answer:  ChatOllama,
    config:      CCConfig,
) -> CompressionTrace:
    """
    Configuration 4: EmbeddingsFilter -- fast, LLM-free compression.

    Making an extra LLM call over each retrieved document is expensive and
    slow. The EmbeddingsFilter provides a cheaper and faster option by
    embedding the documents and query and only returning those documents
    which have sufficiently similar embeddings to the query.

    EmbeddingsFilter mechanics:
        1. Embed the query: q_vec = embed(query)
        2. For each retrieved document D_i:
           d_vec_i = embed(D_i.page_content)
           sim_i   = cosine_similarity(q_vec, d_vec_i)
        3. Keep D_i if sim_i >= similarity_threshold, drop otherwise.
        4. Optionally: keep top-k by similarity score (k parameter).

    Two operating modes (this config uses threshold mode):
        THRESHOLD MODE (similarity_threshold=0.72, k=None):
            All documents above the threshold are returned.
            Variable output size: 0 to BASE_RETRIEVE_K documents.
            Risk: with a strict threshold, may return 0 documents.
            Mitigation: always check output and fall back to raw top-K if empty.
        TOP-K MODE (k=FINAL_K, similarity_threshold=None):
            Always returns exactly K documents, sorted by similarity.
            Fixed output size: always K documents.
            No filtering benefit if K == BASE_RETRIEVE_K.
            Best for: ensuring a minimum context size while still reranking.

    EmbeddingsFilter vs. LLM-based compressors:
        SPEED:      EmbeddingsFilter ~10-50ms (inference only).
                    LLMChainExtractor ~200-800ms (K LLM calls at 200ms each).
        COST:       EmbeddingsFilter: embedding inference only (< $0.0001/query).
                    LLMChainExtractor: K LLM API calls (K * cost_per_call).
        QUALITY:    EmbeddingsFilter: keeps whole documents, no extraction.
                    LLMChainExtractor: extracts only relevant sentences (better precision).
        USE WHEN:   EmbeddingsFilter for high-QPS, latency-sensitive systems.
                    LLMChainExtractor for quality-critical, lower-QPS systems.

    Threshold calibration for BGE-large-en-v1.5:
        BGE-large produces L2-normalized embeddings (norm=1.0).
        Cosine similarity = dot product for normalized vectors.
        Expected similarity distribution on technical domain queries:
            Irrelevant doc: cosine ~ 0.40-0.60
            Loosely relevant: cosine ~ 0.60-0.72
            Clearly relevant: cosine ~ 0.72-0.90
            Near-match: cosine > 0.90
        threshold=0.72 catches "clearly relevant and above" as the filter target.

    Args:
        question    : User query.
        vectorstore : FAISS index.
        bm25        : BM25Plus retriever.
        embeddings  : BGE embeddings (same model used for index and filter).
        llm_answer  : LLM for answer generation (temperature=0.0).
        config      : CCConfig.

    Returns:
        CompressionTrace with fast embedding-based compression metrics.
    """
    trace = CompressionTrace(question=question, strategy="Config4_EmbeddingsFilter")
    t0    = time.perf_counter()

    base_retriever = build_hybrid_base_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw_docs(question, base_retriever, config)
    trace.raw_docs = raw_docs

    # EmbeddingsFilter: threshold mode (k=None means use threshold only)
    emb_filter = EmbeddingsFilter(
        embeddings=embeddings,
        similarity_threshold=config.EMBEDDINGS_FILTER_THRESHOLD,
        # k=None: return all docs above threshold (variable output size)
    )
    cc_retriever = ContextualCompressionRetriever(
        base_compressor=emb_filter,
        base_retriever=base_retriever,
    )

    t_compress           = time.perf_counter()
    compressed_docs      = cc_retriever.invoke(question)
    trace.compression_ms = (time.perf_counter() - t_compress) * 1000

    # Safety: if all docs dropped by strict threshold, fall back to top-K raw docs
    if not compressed_docs:
        logger.warning(
            "Config4: EmbeddingsFilter (threshold=%.2f) dropped all %d docs. "
            "Falling back to raw top-%d.",
            config.EMBEDDINGS_FILTER_THRESHOLD, len(raw_docs), config.FINAL_K,
        )
        compressed_docs = raw_docs[: config.FINAL_K]

    trace.compressed_docs = compressed_docs[: config.FINAL_K]
    trace.compute_compression_metrics()

    logger.info(
        "Config4 EmbeddingsFilter: %d raw -> %d compressed  (%.2fx ratio, %.0fms)",
        len(raw_docs), len(trace.compressed_docs),
        trace.compression_ratio, trace.compression_ms,
    )

    answer, trace.generation_ms = generate_answer(
        question, trace.compressed_docs, llm_answer, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace


In [78]:

# ---------------------------------------------------------------------------
# CONFIG 5: DocumentCompressorPipeline [BEST]
# ---------------------------------------------------------------------------

def run_config5_document_compressor_pipeline(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    embeddings:   HuggingFaceEmbeddings,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       CCConfig,
) -> CompressionTrace:
    """
    Configuration 5: DocumentCompressorPipeline (4-Stage) [BEST].

    The DocumentCompressorPipeline makes it easy to create a pipeline of
    transformations and compressors and run them in sequence. Along with
    compressors we can add BaseDocumentTransformers to our pipeline, which
    don't perform any contextual compression but simply perform some
    transformation on a set of documents.

    4-STAGE PIPELINE DESIGN:

    Stage 1 -- RecursiveCharacterTextSplitter (chunk_size=200, separator="."):
        INPUT:  10 retrieved chunks, each 300-500 chars.
        OUTPUT: ~30-60 sentence-level sub-chunks, each 80-200 chars.
        WHY:    EmbeddingsFilter in Stage 3 operates at the granularity of
                the text unit it receives. If it receives a 450-char chunk
                that is 40% relevant and 60% irrelevant, its embedding is
                a blend of both and may not exceed the threshold (false negative).
                After re-splitting to 200-char sub-chunks, each unit is much
                more semantically homogeneous. A purely irrelevant sentence
                has a low similarity score and gets dropped. A purely relevant
                sentence has a high score and is kept.
                This is the key architectural insight: indexing at chunk granularity
                for recall, filtering at sentence granularity for precision.

    Stage 2 -- EmbeddingsRedundantFilter (similarity_threshold=0.95):
        INPUT:  ~30-60 sentence-level sub-chunks.
        OUTPUT: Deduplicated sub-chunks (near-duplicates removed).
        WHY:    Adjacent chunks at index time (overlap=60) produce many sub-chunks
                that are near-duplicate sentences from the same passage. After
                Stage 1 re-splitting, these duplicates become identical or
                near-identical sub-chunks. EmbeddingsRedundantFilter drops
                sub-chunk B if cosine(embed(A), embed(B)) > 0.95.
                This eliminates the inter-document redundancy problem and
                reduces token budget consumption from overlap duplication.

    Stage 3 -- EmbeddingsFilter (similarity_threshold=0.72):
        INPUT:  Deduplicated sentence-level sub-chunks.
        OUTPUT: Only sub-chunks with cosine > 0.72 to the query.
        WHY:    Fast (no LLM), eliminates the majority of off-topic content.
                Typical reduction: 70-80% of sub-chunks are dropped here.
                This dramatically reduces the input set for Stage 4.
                If Stage 3 keeps 6 sub-chunks instead of 50, Stage 4 (LLM)
                makes 6 extraction calls instead of 50 -- a ~9x cost reduction.

    Stage 4 -- LLMChainExtractor:
        INPUT:  ~4-8 high-relevance sentence-level sub-chunks.
        OUTPUT: Only the verbatim relevant statements from each sub-chunk.
        WHY:    Even a high-cosine sub-chunk may contain some surrounding
                context sentences that are not directly relevant to the query.
                LLMChainExtractor performs the final precision extraction:
                "Given this query, extract only the exact text that answers it."
                After three prior stages of filtering, this final LLM pass
                operates on a tiny, already-filtered document set, making
                it affordable even with per-document LLM calls.

    NET RESULT:
        10 raw chunks (3,500 chars) -> 4-6 sub-chunks (200-400 chars)
        Compression ratio: 10x-20x, making this the highest compression
        configuration. Typical 15x compression observed on research paper corpora.

    Args:
        question      : User query.
        vectorstore   : FAISS index.
        bm25          : BM25Plus retriever.
        embeddings    : BGE embeddings (shared across stages 1-3).
        llm_compress  : LLM for Stage 4 extraction (temperature=0.0).
        llm_answer    : LLM for answer generation (temperature=0.0).
        config        : CCConfig.

    Returns:
        CompressionTrace with full 4-stage compression metrics.
    """
    trace = CompressionTrace(
        question=question, strategy="Config5_DocumentCompressorPipeline [BEST]"
    )
    t0 = time.perf_counter()

    base_retriever = build_hybrid_base_retriever(vectorstore, bm25, config)
    raw_docs, trace.retrieval_ms = retrieve_raw_docs(question, base_retriever, config)
    trace.raw_docs = raw_docs
    logger.info("Config5: %d raw docs retrieved.", len(raw_docs))

    # Stage 1: RecursiveCharacterTextSplitter
    # Using CharacterTextSplitter with period separator for sentence-level splitting.
    # RecursiveCharacterTextSplitter with sentence separators achieves better
    # sentence boundary detection than fixed-char splitting.
    stage1_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.PIPELINE_SPLIT_CHUNK_SIZE,
        chunk_overlap=0,         # no overlap in pipeline splitter -- avoid sub-chunk duplication
        separators=[". ", ".\n", "\n\n", "\n", " "],
        add_start_index=True,
    )

    # Stage 2: EmbeddingsRedundantFilter
    # Uses the same embedding model as the index (BGE-large) for consistency.
    stage2_redundant = EmbeddingsRedundantFilter(
        embeddings=embeddings,
        similarity_threshold=config.REDUNDANCY_THRESHOLD,
    )

    # Stage 3: EmbeddingsFilter
    # k=None: return all sub-chunks above threshold (variable, not fixed top-K).
    # similarity_threshold=0.72: tuned for BGE-large-en-v1.5 on technical text.
    stage3_relevant = EmbeddingsFilter(
        embeddings=embeddings,
        similarity_threshold=config.EMBEDDINGS_FILTER_THRESHOLD,
        k=None,
    )

    # Stage 4: LLMChainExtractor
    # Operates on the small surviving set from Stage 3. Extracts verbatim spans.
    stage4_extractor = LLMChainExtractor.from_llm(llm_compress)

    # Assemble the 4-stage DocumentCompressorPipeline
    pipeline_compressor = DocumentCompressorPipeline(
        transformers=[
            stage1_splitter,    # split to sentence-level sub-chunks
            stage2_redundant,   # remove near-duplicate sub-chunks
            stage3_relevant,    # drop low-similarity sub-chunks (fast)
            stage4_extractor,   # extract verbatim relevant statements (precise)
        ]
    )

    # Wrap base retriever with the pipeline compressor
    cc_retriever = ContextualCompressionRetriever(
        base_compressor=pipeline_compressor,
        base_retriever=base_retriever,
    )

    t_compress           = time.perf_counter()
    compressed_docs      = cc_retriever.invoke(question)
    trace.compression_ms = (time.perf_counter() - t_compress) * 1000

    # Safety: if over-compression (all dropped), fall back to EmbeddingsFilter only
    if not compressed_docs:
        logger.warning(
            "Config5: DocumentCompressorPipeline dropped all docs. "
            "Falling back to EmbeddingsFilter only."
        )
        fallback_filter = EmbeddingsFilter(
            embeddings=embeddings,
            similarity_threshold=config.EMBEDDINGS_FILTER_THRESHOLD - 0.10,
        )
        fallback_cc = ContextualCompressionRetriever(
            base_compressor=fallback_filter,
            base_retriever=base_retriever,
        )
        compressed_docs = fallback_cc.invoke(question)
        if not compressed_docs:
            compressed_docs = raw_docs[: config.FINAL_K]

    trace.compressed_docs = compressed_docs[: config.FINAL_K]
    trace.compute_compression_metrics()

    logger.info(
        "Config5 Pipeline: %d raw -> %d compressed  (%.2fx ratio, %.0fms)",
        len(raw_docs), len(trace.compressed_docs),
        trace.compression_ratio, trace.compression_ms,
    )

    answer, trace.generation_ms = generate_answer(
        question, trace.compressed_docs, llm_answer, config
    )
    trace.final_answer = answer
    trace.total_ms     = (time.perf_counter() - t0) * 1000
    return trace



In [79]:


# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:     str,
    vectorstore:  FAISS,
    bm25:         BM25Retriever,
    embeddings:   HuggingFaceEmbeddings,
    llm_compress: ChatOllama,
    llm_answer:   ChatOllama,
    config:       CCConfig,
) -> Dict[str, CompressionTrace]:
    """Run all five compression configurations on a single question."""
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline": lambda q: run_config1_baseline(
            q, vectorstore, bm25, llm_answer, config),
        "Config2_LLMChainExtractor": lambda q: run_config2_llm_chain_extractor(
            q, vectorstore, bm25, llm_compress, llm_answer, config),
        "Config3_LLMFilter+ListwiseRerank": lambda q: run_config3_llm_filter_listwise(
            q, vectorstore, bm25, llm_compress, llm_answer, config),
        "Config4_EmbeddingsFilter": lambda q: run_config4_embeddings_filter(
            q, vectorstore, bm25, embeddings, llm_answer, config),
        "Config5_DocCompressorPipeline [BEST]": lambda q: run_config5_document_compressor_pipeline(
            q, vectorstore, bm25, embeddings, llm_compress, llm_answer, config),
    }

    traces: Dict[str, CompressionTrace] = {}

    for label, fn in runners.items():
        print(f"\n{'='*60}\nRUNNING: {label}\n{'='*60}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    # Compression summary table
    print("\n" + "=" * 78)
    print("CONTEXTUAL COMPRESSION SUMMARY")
    print(f"{'Config':<45} {'Raw':>5} {'Comp':>5} {'Ratio':>6} "
          f"{'Comp(ms)':>9} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<45} {tr.total_raw_chars:>5,} "
            f"{tr.total_compressed_chars:>5,} "
            f"{tr.compression_ratio:>6.1f}x "
            f"{tr.compression_ms:>9.0f} "
            f"{tr.total_ms:>10.0f}"
        )
    print("=" * 78 + "\n")

    return traces



In [80]:
 """
    End-to-end Contextual Compression Retriever RAG pipeline orchestrator.

    Execution order:
        1.  Download three arXiv PDFs.
        2.  Load and chunk documents (index-time chunk_size=450).
        3.  intialize ollama.
        4.  Initialize two LLM instances:
                llm_compress: temperature=0.0 (deterministic extraction/filtering)
                llm_answer:   temperature=0.0 (deterministic answer generation)
        5.  Load BGE bi-encoder (shared: index, EmbeddingsFilter, RedundantFilter).
        6.  Build/load FAISS index.
        7.  Build/load BM25Plus index.
        8.  Run all five configs on representative demo queries.

    SHARED EMBEDDING MODEL ACROSS COMPRESSION STAGES:
        The same BGE-large-en-v1.5 model is used for:
            - FAISS indexing (document embeddings at index time)
            - FAISS retrieval (query embedding at query time)
            - EmbeddingsRedundantFilter (redundancy detection)
            - EmbeddingsFilter (similarity-based filtering)
        This is critical for calibration consistency: the similarity_threshold=0.72
        is calibrated against BGE-large cosine distributions. Using a different
        model for filtering than for indexing would require re-calibrating all
        threshold values.

    LLM CALL COUNT PER QUERY:
        Config 1: 1 call  (answer only)
        Config 2: BASE_RETRIEVE_K + 1 calls  (K extractions + 1 answer)
        Config 3: BASE_RETRIEVE_K + F + 1 calls (K filter + F listwise + 1 answer)
        Config 4: 1 call  (answer only -- EmbeddingsFilter has zero LLM calls)
        Config 5: S + 1 calls  (S Stage4 extraction calls on survivors + 1 answer)
                  Where S = docs surviving Stage 3 (typically 3-6 out of 10)
    """

'\n   End-to-end Contextual Compression Retriever RAG pipeline orchestrator.\n\n   Execution order:\n       1.  Download three arXiv PDFs.\n       2.  Load and chunk documents (index-time chunk_size=450).\n       3.  intialize ollama.\n       4.  Initialize two LLM instances:\n               llm_compress: temperature=0.0 (deterministic extraction/filtering)\n               llm_answer:   temperature=0.0 (deterministic answer generation)\n       5.  Load BGE bi-encoder (shared: index, EmbeddingsFilter, RedundantFilter).\n       6.  Build/load FAISS index.\n       7.  Build/load BM25Plus index.\n       8.  Run all five configs on representative demo queries.\n\n   SHARED EMBEDDING MODEL ACROSS COMPRESSION STAGES:\n       The same BGE-large-en-v1.5 model is used for:\n           - FAISS indexing (document embeddings at index time)\n           - FAISS retrieval (query embedding at query time)\n           - EmbeddingsRedundantFilter (redundancy detection)\n           - EmbeddingsFilter (simi

In [81]:
config = CCConfig()

In [82]:
pdf_paths   = download_pdfs(config)


2026-05-21 08:43:27 | INFO     | contextual_compression_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-21 08:43:27 | INFO     | contextual_compression_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-21 08:43:27 | INFO     | contextual_compression_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)


In [83]:
chunks = load_and_chunk_documents(pdf_paths, config)

2026-05-21 08:43:30 | INFO     | contextual_compression_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-21 08:43:31 | INFO     | contextual_compression_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-21 08:43:31 | INFO     | contextual_compression_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-21 08:43:31 | INFO     | contextual_compression_rag | Total chunks: 458


In [84]:
llm_compress = build_ollama_llm(config, temperature=config.LLM_COMPRESS_TEMPERATURE)


In [85]:
llm_answer   = build_ollama_llm(config, temperature=config.LLM_ANSWER_TEMPERATURE)


In [86]:
embeddings   = build_bge_embeddings(config)


2026-05-21 08:43:36 | INFO     | contextual_compression_rag | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-21 08:43:36 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-05-21 08:43:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 08:43:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-21 08:43:37 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 08:43:37 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HT

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4321.55it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-21 08:43:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 08:43:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-21 08:43:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-21 08:43:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-21 08:43:41 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-21 08:43:41 | INFO     | httpx |

In [87]:
vectorstore  = build_faiss_index(chunks, embeddings, config)


2026-05-21 08:43:42 | INFO     | contextual_compression_rag | Loading FAISS from './faiss_cc_index' ...
2026-05-21 08:43:42 | INFO     | contextual_compression_rag | FAISS loaded. Vectors: 458


In [88]:
bm25         = build_bm25_index(chunks, config)


2026-05-21 08:43:42 | INFO     | contextual_compression_rag | Loading BM25 from 'bm25_cc_index\bm25plus.pkl' ...


In [89]:
demo_questions = [
        # Broad query -- many loosely relevant chunks; compression should reduce noise
        "How does attention work in the Transformer model?",

        # Precise factual query -- only 1-2 sentences in entire corpus contain the answer
        "What BLEU score did the Transformer base model achieve on WMT 2014 EN-DE translation?",

        # Multi-aspect query -- tests whether pipeline retains all aspects
        "What are the encoder and decoder components in the Transformer architecture?",

        # Domain-specific compression -- technical acronym handling
        "How does BERT use the MLM and NSP pre-training objectives?",

        # Context-dense query -- multiple relevant passages scattered across papers
        "What is the role of the feed-forward network in the Transformer encoder layer?",
    ]

In [90]:
logger.info(
    "Running %d queries across all 5 Compression configurations ...",
    len(demo_questions),
)

for question in demo_questions:
    run_all_configs(
        question, vectorstore, bm25, embeddings, llm_compress, llm_answer, config
    )


2026-05-21 08:43:42 | INFO     | contextual_compression_rag | Running 5 queries across all 5 Compression configurations ...

##############################################################################
QUERY: How does attention work in the Transformer model?
##############################################################################

RUNNING: Config1_Baseline
2026-05-21 08:43:46 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config1_Baseline_NoCompression
Query: How does attention work in the Transformer model?

  Raw docs: 10  (4,164 chars)
  Compressed docs: 4 (1,571 chars)
  Docs dropped: 6
  Compression ratio: 2.65x

  Per-document compression:
    [1] Attention Is All You Need p4: 380 -> 380 chars  (1.0x)
    [2] Attention Is All You Need p1: 444 -> 444 chars  (1.0x)
    [3] Attention Is All You Need p9: 359 -> 359 chars  (1.0x)
    [4] Attention Is All You Need p0: 388 -> 388 chars  (1.0x)

  Latency: retrieval=1202ms  compr

In [91]:
logger.info("=== Contextual Compression Retriever RAG Pipeline Demo Complete ===")


2026-05-21 08:52:46 | INFO     | contextual_compression_rag | === Contextual Compression Retriever RAG Pipeline Demo Complete ===
